# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FarisElbaz/ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The chosen method is LightGBM, this is due to the non-linear interactions shown in the data, along with the LightGBM's fast and less hardware demanding requirements, positioning it better than Linear regression and Deep learning models. The Tree based methods of this model also allow it to run without any manual binning.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

It will be grouped by client using the GroupKfolds.

This was chosen because in a SEO warehouse content URL's from the same domain share domain authority, technical stack, backlink profiles, and crawling cadence. Splitting randomly would cause cross-validation leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
import json
import os
import duckdb
from google.colab import userdata
from huggingface_hub import login
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import GroupKFold

# Ensure directories exist
os.makedirs("work/outputs", exist_ok=True)

# 1. Hugging Face Login & DuckDB Connection
hf_token = userdata.get("hf_copllab_access")
login(token=hf_token)
print("Successfully logged in to Hugging Face Hub!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet"

# 2. Extract Features (Pre-Cutoff) and Honest Forward Target
dataset_sql = f"""
WITH pre_cutoff AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_14d,
        SUM(gsc_clicks) AS clicks_14d,
        AVG(gsc_avg_position) AS avg_pos,
        STDDEV(gsc_avg_position) AS std_pos,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN '2026-03-01' AND '2026-03-07') AS clicks_w1,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN '2026-03-08' AND '2026-03-14') AS clicks_w2,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN '2026-03-01' AND '2026-03-07') AS imp_w1,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN '2026-03-08' AND '2026-03-14') AS imp_w2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
      AND report_date <= '2026-03-14'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
),
post_cutoff AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_forward
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN '2026-03-15' AND '2026-03-28'
    GROUP BY 1, 2
)
SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.imp_14d,
    p.clicks_14d,
    p.avg_pos,
    COALESCE(p.std_pos, 0.0) AS std_pos,
    p.clicks_14d * 1.0 / NULLIF(p.imp_14d, 0) AS observed_ctr,
    p.clicks_w1,
    p.clicks_w2,
    (p.clicks_w2 - p.clicks_w1) * 1.0 / NULLIF(p.clicks_w1, 0) AS click_velocity_delta,
    (p.imp_w2 - p.imp_w1) * 1.0 / NULLIF(p.imp_w1, 0) AS imp_velocity_delta,
    -- Week 4 Baseline Action Score for head-to-head evaluation
    ROUND(
        CASE
            WHEN p.clicks_w1 >= 10 AND (p.clicks_w2 - p.clicks_w1) * 1.0 / p.clicks_w1 <= -0.40
                THEN (p.clicks_w1 - p.clicks_w2) * 2.0
            ELSE p.imp_14d * GREATEST(0.0, 0.035 - (p.clicks_14d * 1.0 / NULLIF(p.imp_14d, 0)))
        END, 2
    ) AS baseline_action_score,
    -- Target: 1 if page captured high traffic or expanded clicks, 0 otherwise
    CASE
        WHEN COALESCE(f.clicks_forward, 0) >= (p.clicks_14d * 1.2) AND COALESCE(f.clicks_forward, 0) >= 10 THEN 1
        WHEN COALESCE(f.clicks_forward, 0) >= 25 THEN 1
        ELSE 0
    END AS target_high_opportunity
FROM pre_cutoff p
LEFT JOIN post_cutoff f
  ON p.client_hash_id = f.client_hash_id
 AND p.content_hash_id = f.content_hash_id;
"""

print("Executing extraction query...")
df = con.sql(dataset_sql).df()
print(
    f"Dataset loaded: {df.shape[0]} rows across {df['client_hash_id'].nunique()} unique clients."
)

# 3. Features & Validation Setup
features = [
    "imp_14d",
    "clicks_14d",
    "avg_pos",
    "std_pos",
    "observed_ctr",
    "clicks_w1",
    "clicks_w2",
    "click_velocity_delta",
    "imp_velocity_delta",
]

df["click_velocity_delta"] = (
    df["click_velocity_delta"].fillna(0.0).clip(-5.0, 5.0)
)
df["imp_velocity_delta"] = df["imp_velocity_delta"].fillna(0.0).clip(-5.0, 5.0)
df["observed_ctr"] = df["observed_ctr"].fillna(0.0)

X = df[features]
y = df["target_high_opportunity"]
groups = df["client_hash_id"]
baseline_scores = df["baseline_action_score"]

# 4. GroupKFold Cross-Validation Loop
gkf = GroupKFold(n_splits=5)
oof_lgb = np.zeros(len(df))
oof_rf = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
  X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
  X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

  # LightGBM Classifier
  model_lgb = lgb.LGBMClassifier(
      n_estimators=150,
      learning_rate=0.05,
      num_leaves=31,
      subsample=0.8,
      colsample_bytree=0.8,
      random_state=42,
      verbose=-1,
  )
  model_lgb.fit(X_tr, y_tr)
  oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]

  # Random Forest Baseline
  model_rf = RandomForestClassifier(
      n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
  )
  model_rf.fit(X_tr.fillna(0), y_tr)
  oof_rf[val_idx] = model_rf.predict_proba(X_va.fillna(0))[:, 1]

# 5. Model vs Baseline Metrics Comparison Table
metrics_summary = [
    {
        "Method": "Week 4 Heuristic Rule (Baseline Score)",
        "ROC-AUC": round(roc_auc_score(y, baseline_scores), 4),
        "PR-AUC (Avg Precision)": round(
            average_precision_score(y, baseline_scores), 4
        ),
    },
    {
        "Method": "Random Forest (Shallow Ensemble)",
        "ROC-AUC": round(roc_auc_score(y, oof_rf), 4),
        "PR-AUC (Avg Precision)": round(average_precision_score(y, oof_rf), 4),
    },
    {
        "Method": "LightGBM (Tuned Boosted Trees)",
        "ROC-AUC": round(roc_auc_score(y, oof_lgb), 4),
        "PR-AUC (Avg Precision)": round(average_precision_score(y, oof_lgb), 4),
    },
]

comparison_df = pd.DataFrame(metrics_summary)
print("\n=== MODEL VS. BASELINE PERFORMANCE (GroupKFold by Client) ===")
print(comparison_df.to_markdown(index=False))

# 6. Feature Importance Receipt
importance_df = pd.DataFrame({
    "feature": features,
    "importance_gain": model_lgb.feature_importances_,
}).sort_values("importance_gain", ascending=False)

print("\n=== LightGBM Top Feature Importances ===")
print(importance_df.to_markdown(index=False))

# Save metrics receipt to git
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(
        {
            "comparison": metrics_summary,
            "top_features": importance_df.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

Successfully logged in to Hugging Face Hub!
Executing extraction query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset loaded: 90593 rows across 40 unique clients.

=== MODEL VS. BASELINE PERFORMANCE (GroupKFold by Client) ===
| Method                                 |   ROC-AUC |   PR-AUC (Avg Precision) |
|:---------------------------------------|----------:|-------------------------:|
| Week 4 Heuristic Rule (Baseline Score) |    0.8114 |                   0.2728 |
| Random Forest (Shallow Ensemble)       |    0.9148 |                   0.5592 |
| LightGBM (Tuned Boosted Trees)         |    0.9175 |                   0.5613 |

=== LightGBM Top Feature Importances ===
| feature              |   importance_gain |
|:---------------------|------------------:|
| std_pos              |               701 |
| avg_pos              |               685 |
| imp_velocity_delta   |               678 |
| imp_14d              |               617 |
| observed_ctr         |               508 |
| clicks_14d           |               460 |
| clicks_w1            |               367 |
| clicks_w2            |   

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The baseline PR-AUC score shows that the average precision was worse than random predictions, with a score of 0.27 whereas its ROC-AUC was 0.81. The lightGBM and Random forest show that tree based ensemble methods are able to somewhat capture and accurately predict the data, all be it mediocerly with precision scores just above random predictions.

The most important features for the LightGBM model are the std_pos and avg_pos showing that steady position alone does not dictate opportunity; high variance in rank position signals an active SERP testing phase by Google, where pages are sensitive to immediate content or metadata interventions.

impression acceleration imp_velocity_delta and imp_14d follow as the 3rd and 4th most important features, showing that impression growth heavily outweighs trailing clicks.

Finally, we can conclude that Click metrics lag as indicators.

Errors could include false positives:

Zero-CLick SERP saturation: The model identifies volatile, high-impression assets in striking range. However, these queries trigger rich search UI elements (direct answer boxes, finanical/unit conversion tables, or dominant video packs) that fulfil search intent directly on the SERP without yielding site visits.

and False negatives:

Exogenous Query Emergence: Pages with low baseline visibility (imp_14d $< 100$) and flat velocity delta that abruptly caught external trend tailwinds (breaking industry news, seasonal shifts) during the post-cutoff window. Historical trailing metrics within the 14-day observation window contain zero signal for these sudden exogenous shifts.

Long-Tail Keyword Expansion: Pages ranking for hundreds of low-volume tail queries that collectively surged in impressions post-cutoff, which historical content-level aggregates smoothed out.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.